# Roadmap agent

Ask which high-ARR Sales deals in Q3 depend on an unreleased Engineering epic.
The agent calls the gateway MCP server as `product_lead` and should choose
`find_related` kind `roadmap_deals`. Each path is a deal, an epic, and the
team that owns the epic.

Leadership-only deals stay hidden. `employee_eng` has no sales or roadmap
grant, so the same question is empty. The agent must not invent those deals.

This notebook needs `OPENROUTER_API_KEY` and a model that can call tools.
`openai/gpt-4o-mini` works. A small model usually does not. Set
`OPENROUTER_MODEL`.

Run [02-seed-sales-product-eng.ipynb](02-seed-sales-product-eng.ipynb) first
if the curated set is not loaded.


In [1]:
import os
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "agents_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import agents_demo

print("model", os.environ.get("OPENROUTER_MODEL", ""))


model anthropic/claude-haiku-5.5


In [2]:
VISIBLE = {
    "deal-northwind-q3",
    "deal-fabrikam-q3",
    "deal-adventure-q3",
    "deal-wideworld-q3",
    "deal-litware-q3",
    "deal-alpine-q3",
}
HIDDEN = {
    "deal-contoso-q3",
    "deal-tailspin-q3",
    "deal-woodgrove-q3",
    "deal-proseware-q3",
    "deal-coho-q2",
    "deal-lucerne-q4",
    "deal-graphic-q3",
}
QUESTION = "Which high-ARR Sales deals in Q3 depend on unreleased Engineering epics?"


def deal_ids(report):
    found = set()
    for row in report["trace"]:
        payload = row["payload"]
        if not isinstance(payload, dict):
            continue
        for path in payload.get("paths") or []:
            for step in path["steps"]:
                if step["type"] == "Deal":
                    found.add(step["id"])
    return found


def calls(report, name):
    return [row for row in report["trace"] if row["tool"] == name]


agent = agents_demo.build_agent("product_lead", agents_demo.ROADMAP_ROLE)
report = agents_demo.run(agent, QUESTION)
agents_demo.print_trace(report["trace"])
related = calls(report, "find_related")
if not related or any(row["args"].get("kind") != "roadmap_deals" for row in related):
    raise SystemExit(f"roadmap agent did not call roadmap_deals: {report['trace']}")
seen = deal_ids(report)
print("product_lead", sorted(seen))
for row in related:
    payload = row["payload"] or {}
    for path in payload.get("paths") or []:
        deal, epic, team = path["steps"]
        print(
            deal["id"],
            deal.get("account_name"),
            deal.get("arr"),
            "->",
            epic["id"],
            epic.get("status"),
            "->",
            team["id"],
        )
        if not deal.get("account_name"):
            raise SystemExit("a deal is missing its account name")
print(report["answer"])
if seen != VISIBLE or seen & HIDDEN:
    raise SystemExit(f"roadmap paths {sorted(seen)}")
missing = sorted(deal for deal in VISIBLE if deal not in report["answer"])
leaked = sorted(deal for deal in HIDDEN if deal in report["answer"])
if missing or leaked:
    raise SystemExit(f"answer missing {missing} leaked {leaked}")


find_related {'kind': 'roadmap_deals', 'quarter': '2026-Q3'} -> 6
product_lead ['deal-adventure-q3', 'deal-alpine-q3', 'deal-fabrikam-q3', 'deal-litware-q3', 'deal-northwind-q3', 'deal-wideworld-q3']
deal-adventure-q3 Adventure Works 250000 -> epic-mobile-offline planned -> product
deal-alpine-q3 Alpine Ski House 275000 -> epic-checkout-3ds in_progress -> payments
deal-fabrikam-q3 Fabrikam 310000 -> epic-identity-scim planned -> identity
deal-litware-q3 Litware 410000 -> epic-notify-webhooks planned -> platform
deal-northwind-q3 Northwind Traders 480000 -> epic-checkout-retry in_progress -> payments
deal-wideworld-q3 Wide World Importers 640000 -> epic-analytics-live in_progress -> data
The roadmap query returned 6 deals in 2026-Q3 that depend on unreleased epics. Each epic is either planned or in progress.

| Deal | Epic | Team |
|---|---|---|
| deal-adventure-q3 | epic-mobile-offline | product |
| deal-alpine-q3 | epic-checkout-3ds | payments |
| deal-fabrikam-q3 | epic-identity-scim

In [3]:
denied_agent = agents_demo.build_agent("employee_eng", agents_demo.ROADMAP_ROLE)
denied = agents_demo.run(denied_agent, QUESTION)
agents_demo.print_trace(denied["trace"])
print(denied["answer"])
related = calls(denied, "find_related")
if not related or any(row["args"].get("kind") != "roadmap_deals" for row in related):
    raise SystemExit(f"denied agent did not call roadmap_deals: {denied['trace']}")
if deal_ids(denied) or any((row["result_count"] or 0) > 0 for row in related):
    raise SystemExit("employee_eng should see no roadmap path")
if any(deal in denied["answer"] for deal in VISIBLE | HIDDEN):
    raise SystemExit(denied["answer"])
print("employee_eng saw nothing")


find_related {'kind': 'roadmap_deals', 'quarter': '2026-Q3'} -> 0
I found no high-ARR deals in Q3 (2026-Q3) that depend on unreleased Engineering epics that I'm allowed to read. The lookup used the default minimum ARR of $200,000, and it returned no paths.
employee_eng saw nothing
